# 05 – Impact table
For each headline metric in August 2026: the value computed with the same definition straight from the raw tables (no cleaning) vs. the clean value, and which data issues explain the difference.
"Raw" = the metric definition applied naively to the exercise_* tables: no deduplication, no test filtering, no price fixes, statuses taken at face value.

## Step 1 – MAU

In [0]:
%sql
-- Raw vs. clean MAU, with the drivers 
WITH raw_events AS (
  -- Same qualifying events as the MAU definition, straight from the raw table
  SELECT user_id, event_timestamp
  FROM workspace.scribd_exercise.exercise_engagement_events
  WHERE event_type IN ('content_open', 'content_complete', 'like', 'share',
                       'comment_post', 'download', 'search_query')
    AND event_timestamp >= TIMESTAMP'2026-08-01'
    AND event_timestamp <  TIMESTAMP'2026-09-01'
),
test_users AS (
  SELECT DISTINCT user_id
  FROM workspace.scribd_exercise.quarantine_users
  WHERE quarantine_reason = 'test_account'
)
SELECT
  (SELECT COUNT(DISTINCT user_id) FROM raw_events) AS raw_mau,
  (SELECT mau FROM workspace.scribd_exercise.metrics_mau
    WHERE month_start = DATE'2026-08-01' AND business_id = 'All businesses') AS clean_mau,
  -- Driver 1: test accounts counted as active users
  (SELECT COUNT(DISTINCT r.user_id) FROM raw_events r
     JOIN test_users t ON r.user_id = t.user_id) AS test_accounts_in_raw;

raw_mau,clean_mau,test_accounts_in_raw
422,401,21


**Results:** 
- August 2026 MAU: raw 422 vs. clean 401. 
- The entire difference (21 users, +5.2%) is test accounts counted as active users. 
- Other issues don't affect MAU: excluded event types are outside the definition, and duplicate users don't change a distinct count.

## Step 2 – Active paid subscribers and MRR
Raw = definitions applied at face value to exercise_subscriptions on 2026-08-31. MRR is shown as a bridge, removing one issue at a time: test accounts, test status, placeholder prices, overlapping subscriptions.

In [0]:
WITH test_users AS (
  SELECT DISTINCT user_id
  FROM workspace.scribd_exercise.quarantine_users
  WHERE quarantine_reason = 'test_account'
),
-- Paid subscriptions "active" on Aug 31 by the definition taken at face value
active_raw AS (
  SELECT s.subscription_id,
         s.user_id,
         s.status,
         s.monthly_price_usd AS raw_price,
         c.monthly_price_usd AS fixed_price,
         CASE WHEN t.user_id IS NOT NULL THEN 1 ELSE 0 END AS is_test_user
  FROM workspace.scribd_exercise.exercise_subscriptions s
  LEFT JOIN test_users t
    ON s.user_id = t.user_id
  LEFT JOIN workspace.scribd_exercise.clean_subscriptions c
    ON s.subscription_id = c.subscription_id
  WHERE s.start_date <= DATE'2026-08-31'
    AND (s.end_date IS NULL OR s.end_date > DATE'2026-08-31')
    AND s.status NOT IN ('paused', 'trialing')
    AND s.plan_tier <> 'Free'
)
SELECT
  -- Active paid subscribers (distinct people)
  (SELECT COUNT(DISTINCT user_id) FROM active_raw) AS raw_paid_subscribers,
  (SELECT active_paid_subscribers FROM workspace.scribd_exercise.metrics_active_subscribers
    WHERE month_end = DATE'2026-08-31' AND business_id = 'All businesses') AS clean_paid_subscribers,

  -- MRR bridge, one issue removed at a time
  ROUND((SELECT SUM(raw_price) FROM active_raw), 2) AS mrr_1_raw,
  ROUND((SELECT SUM(raw_price) FROM active_raw
          WHERE is_test_user = 0), 2) AS mrr_2_without_test_accounts,
  ROUND((SELECT SUM(raw_price) FROM active_raw
          WHERE is_test_user = 0 AND status <> 'TEST_STATUS_DO_NOT_USE'), 2) AS mrr_3_without_test_status,
  ROUND((SELECT SUM(fixed_price) FROM active_raw
          WHERE is_test_user = 0 AND status <> 'TEST_STATUS_DO_NOT_USE'), 2) AS mrr_4_prices_fixed,
  (SELECT mrr_usd FROM workspace.scribd_exercise.metrics_mrr
    WHERE month_end = DATE'2026-08-31' AND business_id = 'All businesses') AS mrr_5_clean_overlaps_removed;

raw_paid_subscribers,clean_paid_subscribers,mrr_1_raw,mrr_2_without_test_accounts,mrr_3_without_test_status,mrr_4_prices_fixed,mrr_5_clean_overlaps_removed
308,285,9245.77,9049.91,8902.04,4129.9,3956.01


**Result:** 
- As of 2026-08-31, raw MRR is $9,245.77 vs. clean $3,956.01: raw overstates MRR 2.3x (+$5,289.76). 
- Bridge: test accounts −$195.86 (#14), test status −$147.87 (#16), placeholder prices −$4,772.14 (#18, 90% of the gap), overlapping subscriptions −$173.89 (#20). 
- Paid subscribers: raw 308 vs. clean 285 (+8%), from test accounts and test-status subscriptions.

## Step 3 – Churn (June–August 2026)
Raw = share of paid subscriptions active on May 31 (at face value) that ended during June–August, straight from exercise_subscriptions. Clean = customers per business who stopped paying (switches are not churn).

In [0]:
WITH test_users AS (
  SELECT DISTINCT user_id
  FROM workspace.scribd_exercise.quarantine_users
  WHERE quarantine_reason = 'test_account'
),
start_raw AS (
  -- Paid subscriptions active on May 31 at face value
  SELECT s.subscription_id, s.user_id, s.business_id, s.end_date,
         CASE WHEN t.user_id IS NOT NULL THEN 1 ELSE 0 END AS is_test_user
  FROM workspace.scribd_exercise.exercise_subscriptions s
  LEFT JOIN test_users t ON s.user_id = t.user_id
  WHERE s.start_date <= DATE'2026-05-31'
    AND (s.end_date IS NULL OR s.end_date > DATE'2026-05-31')
    AND s.status NOT IN ('paused', 'trialing')
    AND s.plan_tier <> 'Free'
)
SELECT
  COUNT(*) AS raw_starting_subscriptions,
  SUM(CASE WHEN end_date BETWEEN DATE'2026-06-01' AND DATE'2026-08-31' THEN 1 ELSE 0 END) AS raw_ended_subscriptions,
  ROUND(SUM(CASE WHEN end_date BETWEEN DATE'2026-06-01' AND DATE'2026-08-31' THEN 1 ELSE 0 END) / COUNT(*), 4) AS raw_churn_rate,
  (SELECT churn_rate FROM workspace.scribd_exercise.metrics_churn
    WHERE window_end = DATE'2026-08-31'
      AND business_id = 'All businesses' AND billing_cycle = 'All billing cycles') AS clean_churn_rate,
  SUM(is_test_user) AS test_account_subscriptions_in_start
FROM start_raw;

raw_starting_subscriptions,raw_ended_subscriptions,raw_churn_rate,clean_churn_rate,test_account_subscriptions_in_start
325,45,0.1385,0.1324,16


**Result:** 
- June–August 2026 churn: raw 13.9% (45 of 325 subscriptions ended) vs. clean 13.2% (38 of 287 customers). 
- Churn is robust to the data issues. Cleaning barely changes churn, because the data problems pull in opposite directions: plan changes look like cancellations in the raw data (making churn look higher), while test accounts (16 subscriptions) and test-status subscriptions add extra customers who never leave (making it look lower)

## Step 4 – Content metrics and session time
Catalog reach and metadata completeness for August 2026, raw vs. clean; average session time over all events, raw vs. clean (issue #25).

In [0]:
WITH raw_catalog AS (
  -- Raw catalog live on Aug 31, including test content
  SELECT content_id, content_type, runtime_minutes_fake
  FROM workspace.scribd_exercise.exercise_content_items
  WHERE publish_date <= DATE'2026-08-31'
),
raw_reached AS (
  -- Items with any content event in August, straight from the raw events
  SELECT DISTINCT content_id
  FROM workspace.scribd_exercise.exercise_engagement_events
  WHERE event_type IN ('content_open', 'content_complete', 'like', 'share', 'comment_post', 'download')
    AND event_timestamp >= TIMESTAMP'2026-08-01'
    AND event_timestamp <  TIMESTAMP'2026-09-01'
)
SELECT
  -- Catalog reach
  (SELECT COUNT(*) FROM raw_catalog) AS raw_live_items,
  ROUND((SELECT COUNT(*) FROM raw_catalog c JOIN raw_reached r ON c.content_id = r.content_id)
        / (SELECT COUNT(*) FROM raw_catalog), 4) AS raw_catalog_reach,
  (SELECT catalog_reach FROM workspace.scribd_exercise.metrics_catalog_reach
    WHERE month_start = DATE'2026-08-01' AND business_id = 'All businesses' AND dimension = 'total') AS clean_catalog_reach,

  -- Metadata completeness (raw: test type and placeholder runtimes count as failures)
  ROUND((SELECT SUM(CASE WHEN content_type <> 'TEST_TYPE'
                          AND runtime_minutes_fake NOT IN (-1, 99999) THEN 1 ELSE 0 END)
         FROM raw_catalog) / (SELECT COUNT(*) FROM raw_catalog), 4) AS raw_metadata_completeness,
  (SELECT pass_rate FROM workspace.scribd_exercise.metrics_metadata_completeness
    WHERE month_start = DATE'2026-08-01' AND business_id = 'All businesses'
      AND requirement = 'All requirements') AS clean_metadata_completeness,

  -- Session time, all events, in minutes
  ROUND((SELECT AVG(session_duration_sec) FROM workspace.scribd_exercise.exercise_engagement_events) / 60, 1) AS raw_avg_session_min,
  ROUND((SELECT AVG(session_duration_sec) FROM workspace.scribd_exercise.clean_engagement_events) / 60, 1) AS clean_avg_session_min,
  ROUND((SELECT PERCENTILE(session_duration_sec, 0.5) FROM workspace.scribd_exercise.clean_engagement_events) / 60, 1) AS clean_median_session_min;

raw_live_items,raw_catalog_reach,clean_catalog_reach,raw_metadata_completeness,clean_metadata_completeness,raw_avg_session_min,clean_avg_session_min,clean_median_session_min
294,0.7823,0.7552,0.9388,0.965,100.1,49.4,49.9


**Result:** 
- Catalog reach: raw 78.2% vs. clean 75.5% (test items and test-account activity make more of the catalog look seen; #4, #14). 
- Metadata completeness: raw 93.9% vs. clean 96.5% (test items counted as catalog items; #4). 
- Average session time over all events: raw 100.1 min vs. clean 49.4 min (median 49.9): placeholder durations of 1 and 7 days double the average (#25).

## Step 5 – Combined impact views
metrics_impact: raw vs. clean for every headline metric (August 2026), with the main driver. metrics_mrr_bridge: the MRR bridge from raw to clean, one row per issue, for a waterfall chart.

In [0]:
-- Impact View 
CREATE OR REPLACE VIEW workspace.scribd_exercise.metrics_impact (
  sort_order COMMENT 'Display order',
  metric COMMENT 'Headline metric',
  raw_value COMMENT 'Value from the raw tables with the same definition and no cleaning (August 2026; churn June–August 2026)',
  clean_value COMMENT 'Value from the clean metric views',
  difference COMMENT 'raw_value minus clean_value',
  pct_difference COMMENT '(raw_value - clean_value) / clean_value',
  unit COMMENT 'Unit of the values',
  main_driver COMMENT 'Data issues that explain the difference (issue numbers from issue_log.md)',
  impact_level COMMENT 'How much cleaning changes the metric, based on the absolute relative difference: High (20% or more), Medium (5% to 20%), Low (under 5%). Thresholds are a judgment call.'
)
COMMENT 'Impact of data cleaning on each headline metric for August 2026: the metric computed straight from the raw tables (same definition, no deduplication, no test filtering, no price fixes, statuses at face value) vs. the clean value, with the main driver of the difference and an impact level.'
AS
WITH raw_mau_events AS (
  SELECT user_id
  FROM workspace.scribd_exercise.exercise_engagement_events
  WHERE event_type IN ('content_open', 'content_complete', 'like', 'share', 'comment_post', 'download', 'search_query')
    AND event_timestamp >= TIMESTAMP'2026-08-01'
    AND event_timestamp <  TIMESTAMP'2026-09-01'
),
raw_subs_aug AS (
  SELECT user_id, monthly_price_usd
  FROM workspace.scribd_exercise.exercise_subscriptions
  WHERE start_date <= DATE'2026-08-31'
    AND (end_date IS NULL OR end_date > DATE'2026-08-31')
    AND status NOT IN ('paused', 'trialing')
    AND plan_tier <> 'Free'
),
raw_subs_may AS (
  SELECT end_date
  FROM workspace.scribd_exercise.exercise_subscriptions
  WHERE start_date <= DATE'2026-05-31'
    AND (end_date IS NULL OR end_date > DATE'2026-05-31')
    AND status NOT IN ('paused', 'trialing')
    AND plan_tier <> 'Free'
),
raw_catalog AS (
  SELECT content_id, content_type, runtime_minutes_fake
  FROM workspace.scribd_exercise.exercise_content_items
  WHERE publish_date <= DATE'2026-08-31'
),
raw_reached AS (
  SELECT DISTINCT content_id
  FROM workspace.scribd_exercise.exercise_engagement_events
  WHERE event_type IN ('content_open', 'content_complete', 'like', 'share', 'comment_post', 'download')
    AND event_timestamp >= TIMESTAMP'2026-08-01'
    AND event_timestamp <  TIMESTAMP'2026-09-01'
),
vals AS (
  SELECT 1 AS sort_order, 'MAU' AS metric,
         CAST((SELECT COUNT(DISTINCT user_id) FROM raw_mau_events) AS DOUBLE) AS raw_value,
         CAST((SELECT mau FROM workspace.scribd_exercise.metrics_mau
                WHERE month_start = DATE'2026-08-01' AND business_id = 'All businesses') AS DOUBLE) AS clean_value,
         'users' AS unit,
         'Test accounts counted as active users (#14)' AS main_driver
  UNION ALL
  SELECT 2, 'Active paid subscribers',
         (SELECT COUNT(DISTINCT user_id) FROM raw_subs_aug),
         (SELECT active_paid_subscribers FROM workspace.scribd_exercise.metrics_active_subscribers
           WHERE month_end = DATE'2026-08-31' AND business_id = 'All businesses'),
         'people',
         'Test accounts and test-status subscriptions counted as active (#14, #16)'
  UNION ALL
  SELECT 3, 'MRR',
         (SELECT SUM(monthly_price_usd) FROM raw_subs_aug),
         (SELECT mrr_usd FROM workspace.scribd_exercise.metrics_mrr
           WHERE month_end = DATE'2026-08-31' AND business_id = 'All businesses'),
         'USD per month',
         'Placeholder prices of 999.99 (#18, about 90% of the gap); also test accounts, test status and overlapping subscriptions (#14, #16, #20)'
  UNION ALL
  SELECT 4, 'Churn rate (3 months, Jun–Aug)',
         (SELECT SUM(CASE WHEN end_date BETWEEN DATE'2026-06-01' AND DATE'2026-08-31' THEN 1 ELSE 0 END) / COUNT(*)
            FROM raw_subs_may),
         (SELECT churn_rate FROM workspace.scribd_exercise.metrics_churn
           WHERE window_end = DATE'2026-08-31' AND business_id = 'All businesses'
             AND billing_cycle = 'All billing cycles'),
         'share of starting customers',
         'Little change: plan changes look like cancellations in raw data (pushes it up); test accounts and test-status subscriptions add customers who never leave (pushes it down)'
  UNION ALL
  SELECT 5, 'Catalog reach',
         (SELECT COUNT(*) FROM raw_catalog c JOIN raw_reached r ON c.content_id = r.content_id)
           / (SELECT COUNT(*) FROM raw_catalog),
         (SELECT catalog_reach FROM workspace.scribd_exercise.metrics_catalog_reach
           WHERE month_start = DATE'2026-08-01' AND business_id = 'All businesses' AND dimension = 'total'),
         'share of live catalog',
         'Raw catalog has 294 live items vs. 286 clean: the 8 test items (#4) were all engaged with, and test-account activity (#14) makes more of the catalog look seen'
  UNION ALL
  SELECT 6, 'Metadata completeness',
         (SELECT SUM(CASE WHEN content_type <> 'TEST_TYPE' AND runtime_minutes_fake NOT IN (-1, 99999) THEN 1 ELSE 0 END)
            FROM raw_catalog) / (SELECT COUNT(*) FROM raw_catalog),
         (SELECT pass_rate FROM workspace.scribd_exercise.metrics_metadata_completeness
           WHERE month_start = DATE'2026-08-01' AND business_id = 'All businesses'
             AND requirement = 'All requirements'),
         'share of live catalog',
         'Raw catalog has 294 live items vs. 286 clean: the 8 test items (#4) count as items with an invalid type'
)
SELECT sort_order,
       metric,
       ROUND(raw_value, 4) AS raw_value,
       ROUND(clean_value, 4) AS clean_value,
       ROUND(raw_value - clean_value, 4) AS difference,
       ROUND((raw_value - clean_value) / clean_value, 4) AS pct_difference,
       unit,
       main_driver,
       CASE
         WHEN ABS((raw_value - clean_value) / clean_value) >= 0.20 THEN 'High'
         WHEN ABS((raw_value - clean_value) / clean_value) >= 0.05 THEN 'Medium'
         ELSE 'Low'
       END AS impact_level
FROM vals;

In [0]:
-- MRR bridge view
CREATE OR REPLACE VIEW workspace.scribd_exercise.metrics_mrr_bridge (
  step_order COMMENT 'Order of the steps, from raw to clean',
  step COMMENT 'What is removed or fixed at this step',
  mrr_after_step COMMENT 'MRR after this step (USD per month, as of 2026-08-31)',
  change_usd COMMENT 'Change from the previous step: the amount this issue adds to raw MRR',
  issue COMMENT 'Issue number from issue_log.md'
)
COMMENT 'MRR bridge as of 2026-08-31: from raw MRR (definitions at face value on the raw table) to clean MRR, removing one data issue at a time. The order is a choice; a different order splits the amounts slightly differently, but the total is the same. For a waterfall chart.'
AS
WITH test_users AS (
  SELECT DISTINCT user_id
  FROM workspace.scribd_exercise.quarantine_users
  WHERE quarantine_reason = 'test_account'
),
active_raw AS (
  SELECT s.status,
         s.monthly_price_usd AS raw_price,
         c.monthly_price_usd AS fixed_price,
         CASE WHEN t.user_id IS NOT NULL THEN 1 ELSE 0 END AS is_test_user
  FROM workspace.scribd_exercise.exercise_subscriptions s
  LEFT JOIN test_users t ON s.user_id = t.user_id
  LEFT JOIN workspace.scribd_exercise.clean_subscriptions c ON s.subscription_id = c.subscription_id
  WHERE s.start_date <= DATE'2026-08-31'
    AND (s.end_date IS NULL OR s.end_date > DATE'2026-08-31')
    AND s.status NOT IN ('paused', 'trialing')
    AND s.plan_tier <> 'Free'
),
steps AS (
  SELECT 1 AS step_order, 'Raw MRR' AS step,
         (SELECT SUM(raw_price) FROM active_raw) AS mrr_after_step, '' AS issue
  UNION ALL
  SELECT 2, 'Remove test accounts',
         (SELECT SUM(raw_price) FROM active_raw WHERE is_test_user = 0), '#14'
  UNION ALL
  SELECT 3, 'Remove test-status subscriptions',
         (SELECT SUM(raw_price) FROM active_raw
           WHERE is_test_user = 0 AND status <> 'TEST_STATUS_DO_NOT_USE'), '#16'
  UNION ALL
  SELECT 4, 'Fix placeholder prices',
         (SELECT SUM(fixed_price) FROM active_raw
           WHERE is_test_user = 0 AND status <> 'TEST_STATUS_DO_NOT_USE'), '#18'
  UNION ALL
  SELECT 5, 'Count one subscription per customer and business (clean MRR)',
         (SELECT mrr_usd FROM workspace.scribd_exercise.metrics_mrr
           WHERE month_end = DATE'2026-08-31' AND business_id = 'All businesses'), '#20'
)
SELECT step_order,
       step,
       ROUND(mrr_after_step, 2) AS mrr_after_step,
       ROUND(mrr_after_step - LAG(mrr_after_step) OVER (ORDER BY step_order), 2) AS change_usd,
       issue
FROM steps;

In [0]:
SELECT metric, raw_value, clean_value, difference, pct_difference, impact_level
FROM workspace.scribd_exercise.metrics_impact
ORDER BY sort_order;

metric,raw_value,clean_value,difference,pct_difference,impact_level
MAU,422.0,401.0,21.0,0.0524,Medium
Active paid subscribers,308.0,285.0,23.0,0.0807,Medium
MRR,9245.77,3956.01,5289.76,1.3371,High
"Churn rate (3 months, Jun–Aug)",0.1385,0.1324,0.0061,0.0458,Low
Catalog reach,0.7823,0.7552,0.0271,0.0359,Low
Metadata completeness,0.9388,0.965,-0.0262,-0.0272,Low


In [0]:
SELECT * FROM workspace.scribd_exercise.metrics_mrr_bridge ORDER BY step_order;

step_order,step,mrr_after_step,change_usd,issue
1,Raw MRR,9245.77,null,
2,Remove test accounts,9049.91,-195.86,#14
3,Remove test-status subscriptions,8902.04,-147.87,#16
4,Fix placeholder prices,4129.9,-4772.14,#18
5,Count one subscription per customer and business (clean MRR),3956.01,-173.89,#20
